# IceNet-MP Pipeline Demo

This notebook demonstrates the current IceNet-MP machine-learning pipeline through the `imp` command-line interface while explaining the design choices behind it.

**Target audience:** developers and contributors who want to understand how the pipeline is configured, run, inspected, and adapted.

**You'll learn how to:**
- run dataset creation, training, and evaluation end-to-end;
- inspect the artifacts produced by a run;
- understand the standalone and Encode-Process-Decode model patterns;
- switch model and training configurations without editing Python code;
- understand how Hydra composes data, model, logger, training, evaluation, prediction, and platform settings;
- see how multimodal real observations fit into the same architecture.

The **default walkthrough is account-free**. It uses IceNet-MP's generated synthetic sea-ice dataset and local-file logger, so it does not require a Copernicus CDS account or a Weights & Biases account. A real-data route is documented separately at the end.


## Notebook Structure

[**Section 1: End-to-End Training Pipeline**](#section-1-end-to-end-training-pipeline)
- Generate a small synthetic dataset locally.
- Train the current `quick_test` model for one epoch.
- Find the produced checkpoint and saved configuration.
- Evaluate the checkpoint and inspect local artifacts.

[**Section 2: Model Flexibility**](#section-2-model-flexibility)
- Compare standalone models with the Encode-Process-Decode architecture.
- Inspect the current U-Net-based `quick_test` model.
- See the current alternative model configurations.
- Run the persistence baseline on the same synthetic data.
- Relate the single-source synthetic example to the multimodal real-data pipeline.

[**Section 3: Full Flexibility (Advanced Example)**](#section-3-full-flexibility-advanced-example)
- Inspect the Hydra configuration hierarchy.
- Understand data, model, training, prediction, logger, and evaluation configuration groups.
- Use command-line overrides and local config files for reproducible experiments.
- See how Anemoi-backed real datasets differ from the generated synthetic dataset.

[**Optional Real-Data Route**](#optional-real-data-route)
- Use the maintained `demo_notebook` configuration with ARGO, OSISAF, and ERA5 data.
- Understand the CDS and W&B prerequisites for that route.


## Setup

From the repository root, install the notebook dependency group and start Jupyter from the `notebooks` directory:

```bash
uv sync --group notebooks
cd notebooks
uv run jupyter lab
```

Starting Jupyter from `notebooks/` keeps the repository-relative paths used by the shipped configurations consistent. The `imp` executable comes from the project environment created by `uv`.


In [ ]:
from pathlib import Path
import time

notebooks_dir = Path.cwd().resolve()
if notebooks_dir.name != "notebooks":
    raise RuntimeError(
        "Start Jupyter from the repository's notebooks directory before running this demo."
    )

repo_root = notebooks_dir.parent
base_path = (notebooks_dir / "../base").resolve()

print(f"Repository: {repo_root}")
print(f"Default synthetic data/run root: {base_path}")


### Environment verification

The `imp` entry point should expose dataset, training, evaluation, and other application commands. This confirms that the notebook kernel is using the project environment.


In [ ]:
!imp --help


# Section 1: End-to-End Training Pipeline

The original demo used real OSISAF and ERA5 data and therefore required external services before a new contributor could run the first example. The current repository includes a purpose-built `synthetic` root configuration that is better suited to an introductory executable path.

The synthetic configuration still exercises the real IceNet-MP pipeline. It changes the inputs and logging destination:

- `data=synthetic` generates a small moving sea-ice pattern locally;
- `evaluate=synthetic` uses the lightweight evaluation profile;
- `loggers=local_files` writes metrics and media to files rather than W&B;
- `predict=synthetic-2d` predicts two future steps from three history steps;
- `train=synthetic` uses the normal training machinery with synthetic-specific callbacks.

The purpose of this section is to verify and understand the pipeline, not to produce a scientifically meaningful sea-ice forecast.


### Inspect the root synthetic configuration


In [ ]:
synthetic_root = repo_root / "icenet_mp/config/synthetic.yaml"
print(synthetic_root.read_text())


### Create and inspect the synthetic dataset

`imp datasets create` builds every dataset selected by the root configuration. For `synthetic`, the configured source is generated locally rather than downloaded from ERA5 or OSISAF. Rerunning the command can reuse an existing generated dataset unless overwrite is requested.


In [ ]:
!imp datasets create --config-name=synthetic


In [ ]:
!imp datasets inspect --config-name=synthetic


### Understand the synthetic sample

The synthetic data configuration selects one generated sea-ice dataset and a synthetic train/validation/test split. The prediction configuration uses `sic-ssmis` as the target group, three history steps, and two forecast steps.

This deliberately small example keeps the CLI workflow representative while avoiding large external downloads.


In [ ]:
for path in [
    repo_root / "icenet_mp/config/data/synthetic.yaml",
    repo_root / "icenet_mp/config/data/split/synthetic.yaml",
    repo_root / "icenet_mp/config/predict/synthetic-2d.yaml",
]:
    print(f"\n--- {path.relative_to(repo_root)} ---")
    print(path.read_text())


### Train the learned model

The synthetic root configuration currently uses the `quick_test` model inherited from `base.yaml`. It is an Encode-Process-Decode model with naive rescaling encoders, a U-Net processor, and a naive decoder.

The normal synthetic training configuration permits up to 30 epochs. For this notebook we override it to **one epoch** so the example remains a practical pipeline demonstration.


In [ ]:
learned_training_started = time.time()
!imp train --config-name=synthetic train.trainer.max_epochs=1


### Locate the run artifacts

When W&B is not present, IceNet-MP creates a local run directory under:

`<base_path>/training/local/run-<timestamp>-<id>/`

The run contains its checkpoint(s) and `files/model_config.yaml`, which records the composed configuration used for the run. The local-file logger writes metrics and plotting artifacts under `<base_path>/report/`.

The cell below finds the checkpoint created by the training command above rather than relying on a hard-coded epoch, run ID, or personal filesystem path.


In [ ]:
checkpoint_candidates = [
    path
    for path in (base_path / "training" / "local").glob("**/checkpoints/*.ckpt")
    if path.stat().st_mtime >= learned_training_started - 2
]
if not checkpoint_candidates:
    raise FileNotFoundError(
        "No checkpoint created by the one-epoch synthetic training run was found."
    )

learned_checkpoint = max(checkpoint_candidates, key=lambda path: path.stat().st_mtime)
learned_run_dir = learned_checkpoint.parent.parent
learned_config = learned_run_dir / "files" / "model_config.yaml"

print(f"Checkpoint: {learned_checkpoint}")
print(f"Saved config: {learned_config}")
print(f"Local report directory: {base_path / 'report'}")


### Evaluate the learned model

Evaluation loads the trained checkpoint, reconstructs the model configuration, and runs the configured test split. Using the same `synthetic` root configuration keeps the data and local logging route consistent with training.


In [ ]:
!imp evaluate --config-name=synthetic --checkpoint="{learned_checkpoint}"


### Inspect local outputs

The saved `model_config.yaml` is the most useful reproducibility artifact because it captures the composed configuration used for the run. The local logger also writes metrics and any configured images/videos without requiring a cloud service.


In [ ]:
report_dir = base_path / "report"

print("Saved model config exists:", learned_config.exists())
print("Metrics file exists:", (report_dir / "metrics.jsonl").exists())

metrics_path = report_dir / "metrics.jsonl"
if metrics_path.exists():
    recent_metrics = metrics_path.read_text().splitlines()[-5:]
    print("\nRecent metric records:")
    for line in recent_metrics:
        print(line)

for media_name in ("images", "videos"):
    media_dir = report_dir / media_name
    count = len(list(media_dir.glob("*"))) if media_dir.exists() else 0
    print(f"{media_name}: {count} file(s)")


# Section 2: Model Flexibility

IceNet-MP supports two useful model patterns.

A **standalone model** receives the configured input and produces a prediction directly. Persistence is the clearest example: its prediction is simply based on the most recent target state, so it does not need learned encoders, a processor, or a decoder.

An **Encode-Process-Decode model** first maps each configured input dataset into a common latent representation. Those latent inputs are concatenated, processed by a core model, then decoded into the target output space. This separation makes it possible to combine sources with different channel counts and spatial grids and to change the processor independently of the data interfaces.


### Standalone model pattern

![Standalone pipeline](../docs/src/assets/pipeline-standalone.png)

Standalone models are conceptually simple and are useful for baselines or architectures whose input/output handling is built directly into the model. Their trade-off is that changing the set or geometry of inputs can require model-specific changes.


### Encode-Process-Decode pattern

![Encode-Process-Decode pipeline](../docs/src/assets/pipeline-encode-process-decode.png)

For the current `EncodeProcessDecode` implementation:

1. every input dataset gets its own configured encoder;
2. encoder outputs share a latent spatial shape;
3. the encoded inputs are concatenated along the channel dimension;
4. the processor forecasts in latent space;
5. the decoder maps the result into the configured target space.

This is the architectural basis for combining sources such as sea-ice concentration, ERA5 weather variables, and ARGO float observations.


## The current `quick_test` model

The default synthetic example uses `model=quick_test`. Its model config defines naive linear/rescaling encoders for the supported input groups, a `UNetProcessor`, and a naive decoder. On the synthetic route only the configured `sic-ssmis` source is instantiated; on a multimodal route the same model definition can instantiate additional source-specific encoders.


In [ ]:
quick_test_config = repo_root / "icenet_mp/config/model/quick_test.yaml"
print(quick_test_config.read_text())


## Alternative learned model configurations

The original notebook demonstrated model switching with names that have since been replaced. The same design principle remains current: model choice is a Hydra configuration group, so the pipeline code does not need to be edited to select another architecture.

Current model configurations include larger U-Net variants, CNN encoder/decoder combinations, a CNN-ViT-CNN model, diffusion-based models, piecewise models, and persistence. They do **not** all have identical compute or shape requirements, so the tiny synthetic demo should not be treated as a universal benchmark for every model config.


In [ ]:
model_config_dir = repo_root / "icenet_mp/config/model"
print("Available model configs:")
for path in sorted(model_config_dir.glob("*.yaml")):
    print(" -", path.stem)


## Standalone persistence model

Persistence is a useful baseline because it answers a simple question: does a learned model improve on carrying the latest observed target state forward?

The integrated persistence model is selected with `model=persistence`. Its dedicated `train=persistence` profile runs for one epoch and uses an unconditional checkpoint callback so that the normal pipeline can still produce a checkpoint/config artifact even though there are no learned parameters to optimise.

Using the same synthetic data makes the comparison structurally fair: only the model/training configuration changes.


In [ ]:
persistence_training_started = time.time()
!imp train --config-name=synthetic model=persistence train=persistence


In [ ]:
persistence_candidates = [
    path
    for path in (base_path / "training" / "local").glob("**/checkpoints/*.ckpt")
    if path.stat().st_mtime >= persistence_training_started - 2
]
if not persistence_candidates:
    raise FileNotFoundError("No persistence checkpoint was created.")

persistence_checkpoint = max(
    persistence_candidates, key=lambda path: path.stat().st_mtime
)
print(f"Persistence checkpoint: {persistence_checkpoint}")


In [ ]:
!imp evaluate --config-name=synthetic --checkpoint="{persistence_checkpoint}" model=persistence train=persistence


The learned one-epoch run and persistence run are both demonstration-scale experiments. Their evaluation output can be compared to understand the baseline relationship, but neither should be interpreted as a production skill assessment.


## Encoders and multimodality

The account-free synthetic route is intentionally single-source. The real `sample_south` data configuration shows the multimodal structure the project is designed for. It composes three configured datasets:

- ARGO float observations (`float-argo`);
- OSISAF sea-ice concentration (`sic-osisaf`);
- ERA5 atmospheric data (`era5`).

`EncodeProcessDecode` creates one encoder for each input space and concatenates their latent outputs before processing. This lets data with different native grids and channel counts contribute to a common forecasting model.


In [ ]:
sample_south_config = repo_root / "icenet_mp/config/data/sample_south.yaml"
print(sample_south_config.read_text())


# Section 3: Full Flexibility (Advanced Example)

The pipeline is controlled through Hydra configuration rather than hard-coded experiment choices. A root config composes several configuration groups, and command-line overrides can replace a whole group or one nested value.

This arrangement serves two purposes:

- **flexibility:** experiments can change data, models, loss functions, training profiles, loggers, prediction horizons, or platforms without editing application code;
- **reproducibility:** the fully composed run configuration is saved alongside checkpoints and can be inspected later.


## Root configuration composition

`base.yaml` defines the normal real-data defaults. Among other choices it selects `data=sample_south`, the W&B logger, `model=quick_test`, the standard training/evaluation profiles, and the default platform.

`synthetic.yaml` inherits from `base` but deliberately overrides the groups needed for a self-contained local demonstration. This is a useful example of Hydra composition: the application remains the same while the experiment definition changes.


In [ ]:
for path in [
    repo_root / "icenet_mp/config/base.yaml",
    repo_root / "icenet_mp/config/synthetic.yaml",
]:
    print(f"\n--- {path.relative_to(repo_root)} ---")
    print(path.read_text())


### Browse the current configuration groups

The old notebook contained a recorded directory listing that quickly became stale. The cell below generates the list from the checked-out repository instead. Each group can contain alternatives that are selected from a root config or through a CLI override.


In [ ]:
for group in [
    "data",
    "model",
    "loss",
    "loggers",
    "train",
    "evaluate",
    "predict",
    "platform",
]:
    directory = repo_root / "icenet_mp/config" / group
    names = [path.stem for path in sorted(directory.glob("*.yaml"))]
    print(f"{group}: {', '.join(names)}")


## Hydra overrides

A root configuration can be refined directly on the command line. Examples already used in this notebook are:

```bash
# Override one nested value
imp train --config-name=synthetic train.trainer.max_epochs=1

# Replace the model and training configuration groups
imp train --config-name=synthetic model=persistence train=persistence
```

Group overrides are powerful, but compatibility still matters. A different model may require a different latent grid, loss, training profile, or compute budget. The baseline root configs in `icenet_mp/config/baseline/` demonstrate supported combinations for larger experiments.


## Local experiment configs

For repeated local work, create an ignored `*.local.yaml` file in `icenet_mp/config/` rather than modifying a tracked configuration. The repository `.gitignore` excludes these local configs.

For example, a local experiment can inherit the account-free synthetic route and change only the data/run location and epoch count:

```yaml
# icenet_mp/config/my_experiment.local.yaml
defaults:
  - synthetic
  - _self_

base_path: ../my_experiment

train:
  trainer:
    max_epochs: 5
```

It can then be selected with:

```bash
imp train --config-name=my_experiment.local
```

For real or HPC experiments, use the repository's configuration guide and appropriate platform/data overrides rather than copying machine-specific paths into a notebook.


## Creating and inspecting datasets

Dataset creation uses the configured ingestion sources and Anemoi dataset machinery. The root `data` group selects one or more dataset definitions plus a train/validation/test split.

For the synthetic route, the selected dataset uses IceNet-MP's synthetic source and therefore needs no external download credentials. Real ERA5 definitions use the CDS API; OSISAF and ARGO have their own configured sources.

The same CLI applies to either route:

```bash
imp datasets create --config-name=<root-config>
imp datasets inspect --config-name=<root-config>
```

This separation between dataset definition and command implementation is why the account-free synthetic demo can exercise the same dataset CLI as the real pipeline.


In [ ]:
synthetic_dataset_config = (
    repo_root
    / "icenet_mp/config/data/datasets/samp_sicglobal_synthetic_5p625_2021_2024_24h_v2.yaml"
)
print(synthetic_dataset_config.read_text())


## Training and prediction configuration

Training behaviour is also composed from configuration. `train/synthetic.yaml` inherits the standard training profile, selects callbacks appropriate to the synthetic workflow, and sets a 30-epoch maximum. The notebook's one-epoch run is only a CLI override.

Prediction configuration specifies the target dataset group/variables and the number of history and forecast steps. For the synthetic demo this is three history steps and two forecast steps.


In [ ]:
for path in [
    repo_root / "icenet_mp/config/train/synthetic.yaml",
    repo_root / "icenet_mp/config/train/persistence.yaml",
    repo_root / "icenet_mp/config/predict/synthetic-2d.yaml",
]:
    print(f"\n--- {path.relative_to(repo_root)} ---")
    print(path.read_text())


## Logging and artifacts

Logging is a configuration choice rather than a requirement of the model code.

- `loggers/local_files.yaml` uses `LocalFileLogger`, which writes metrics and media locally. This is what the synthetic root config selects.
- `loggers/wandb.yaml` uses Lightning's `WandbLogger` for normal tracked experiments. Standard real-data configurations select it by default.

Regardless of cloud logging, IceNet-MP saves the composed `model_config.yaml` in the local run directory alongside checkpoints. That file is the key record of the actual experiment configuration.


In [ ]:
for path in [
    repo_root / "icenet_mp/config/loggers/local_files.yaml",
    repo_root / "icenet_mp/config/loggers/wandb.yaml",
]:
    print(f"\n--- {path.relative_to(repo_root)} ---")
    print(path.read_text())


## Evaluating a model

`imp evaluate` reconstructs a trained model from its checkpoint and runs the configured test split. Evaluation callbacks control additional outputs.

The standard `evaluate=default` profile includes activation capture, metric summary, and plotting. The synthetic root configuration uses `evaluate=synthetic`, which inherits the default profile but reduces the callback set to plotting for a lightweight demonstration.

Plotting can produce static maps and videos for selected test batches. `MetricSummaryCallback` provides summary metrics for configurations that include it. More specialised diagnostics can enable `ActivationSaver` and use `--save-layer`, as demonstrated by the separate diagnostics notebook.


In [ ]:
for path in [
    repo_root / "icenet_mp/config/evaluate/default.yaml",
    repo_root / "icenet_mp/config/evaluate/synthetic.yaml",
    repo_root / "icenet_mp/config/evaluate/callbacks/plotting.yaml",
    repo_root / "icenet_mp/config/evaluate/callbacks/metric_summary.yaml",
]:
    print(f"\n--- {path.relative_to(repo_root)} ---")
    print(path.read_text())


## Reproducing a run

The original notebook tied reproducibility closely to W&B. W&B remains useful for tracked experiments, but reproducibility does not depend on having a W&B account.

For any local run, inspect the saved `files/model_config.yaml` next to the checkpoint. It contains the composed configuration after root defaults and overrides have been applied. A useful workflow is:

1. keep the checkpoint and its saved `model_config.yaml` together;
2. copy/adapt that configuration into an ignored `*.local.yaml` file when reproducing or extending the experiment;
3. record any deliberate CLI overrides;
4. use the same dataset definitions/splits when making comparisons.


In [ ]:
if learned_config.exists():
    print(learned_config.read_text())
else:
    print("Run the learned-model training section first to create model_config.yaml.")


# Optional Real-Data Route

The repository still contains `icenet_mp/config/demo_notebook.yaml`. It is **not** the account-free entry point.

`demo_notebook.yaml` inherits `base.yaml`, limits training to 10 epochs, sets `base_path: ../my_data`, and leaves the normal real-data composition in place. Through `base.yaml` this selects `data=sample_south` and the W&B logger. `sample_south` contains ARGO, OSISAF sea-ice concentration, and ERA5 weather data.

Because ERA5 is obtained through the CDS API, dataset creation requires a configured Copernicus Climate Data Store account/API key. Standard training and evaluation use W&B unless you deliberately configure W&B offline or use another supported logger in a suitable local config.

After setting up the required services, the real demo route is:

```bash
imp datasets create --config-name=demo_notebook
imp datasets inspect --config-name=demo_notebook
imp train --config-name=demo_notebook
# Then evaluate a checkpoint produced by that training run:
imp evaluate --config-name=demo_notebook --checkpoint=/path/to/checkpoint.ckpt
```

For a shorter real-data smoke test, the epoch count can be overridden without editing YAML:

```bash
imp train --config-name=demo_notebook train.trainer.max_epochs=1
```

These commands are documented here rather than executed automatically so that the default notebook remains usable without external accounts or multi-gigabyte real-data downloads.


In [ ]:
for path in [
    repo_root / "icenet_mp/config/demo_notebook.yaml",
    repo_root / "icenet_mp/config/data/sample_south.yaml",
]:
    print(f"\n--- {path.relative_to(repo_root)} ---")
    print(path.read_text())


## What to use next

- `layer_diagnostics.ipynb` is maintained as an advanced activation-capture workflow for the current UNet/`quick_test` model. It requires a compatible checkpoint and existing real datasets, so it remains separate from this account-free default walkthrough.
- `ARGO_data.ipynb` demonstrates downloading and gridding real non-gridded ARGO observations.
- `case_study_whale_corridors.ipynb` is retained as a whale/shipping case-study artifact with its own data prerequisites.
- `extract_anomalies.ipynb` followed by `degrid_and_visualise.ipynb` provides the older controlled synthetic non-gridded research workflow.

The notebook inventory, maintenance decisions, and recommended ordering are summarised in `notebooks/README.md` and the user-guide notebook page.


## Summary

This walkthrough uses the same current IceNet-MP CLI and configuration system across generated and real data. The account-free synthetic path is intended to make the complete mechanics easy to run, while the real-data configuration demonstrates how the same architecture scales to multimodal observations and external data services.

The important design idea is that data sources, encoders/processors/decoders, training behaviour, logging, prediction horizons, evaluation callbacks, and platforms are all composed through configuration. That makes experiments adaptable without duplicating pipeline code and gives each run a configuration artifact that can be inspected and reproduced.
